In [ ]:
# CASE STUDIES: FLAGGED TRANSACTIONS

print("\n" + "=" * 70)
print("CASE STUDIES - HIGH-RISK TRANSACTIONS IDENTIFIED")
print("=" * 70)

# Get full dataset with predictions
full_df = df_final.copy()

# Get high-risk transactions (anomaly probability > 0.6)
high_risk = full_df[full_df['anomaly_probability'] > 0.6].sort_values('anomaly_probability', ascending=False)

print("\n" + "-" * 70)
print("TOP 10 HIGHEST RISK TRANSACTIONS")
print("-" * 70)

# Select columns for display
case_study_cols = [
    'TransactionID', 'TransactionAmount', 'LoginAttempts', 'Channel', 
    'Location', 'anomaly_probability', 'TransactionDate'
]

# Format for display
for idx, row in high_risk.head(10).iterrows():
    risk_factors = []
    if row['LoginAttempts'] > 1:
        risk_factors.append(f"{row['LoginAttempts']} login attempts")
    if row['TransactionAmount'] > 500:
        risk_factors.append(f"High amount: ${row['TransactionAmount']:.2f}")
    if row['Channel'] == 'Online':
        risk_factors.append("Online channel")
    
    print(f"""
┌─────────────────────────────────────────────────────────────────┐
│ Transaction: {row['TransactionID']}                    Risk: {row['anomaly_probability']:.1%} │
├─────────────────────────────────────────────────────────────────┤
│ Amount:       ${row['TransactionAmount']:>8.2f}    Channel:    {row['Channel']:<10}             │
│ Login Atts:   {row['LoginAttempts']:>2}x               Location:   {row['Location']:<15}          │
│ Date:         {str(row['TransactionDate'])[:10]}                                    │
├─────────────────────────────────────────────────────────────────┤
│ RISK FACTORS:                                                     │
│  {' | '.join(risk_factors) if risk_factors else 'Statistical anomaly pattern'}                                 │
├─────────────────────────────────────────────────────────────────┤
│ RECOMMENDED ACTION: {'BLOCK - High fraud risk' if row['anomaly_probability'] > 0.7 else 'REVIEW - Verify with customer'}                    │
└─────────────────────────────────────────────────────────────────┘""")

print("\n" + "-" * 70)
print("ACCOUNT-LEVEL RISK SUMMARY")
print("-" * 70)

# Find accounts with multiple flagged transactions
account_risk = full_df[full_df['is_anomaly'] == 1].groupby('AccountID').agg({
    'TransactionID': 'count',
    'anomaly_probability': 'mean',
    'TransactionAmount': 'sum'
}).rename(columns={'TransactionID': 'Flagged_Count', 'anomaly_probability': 'Avg_Risk', 'TransactionAmount': 'Total_Amount'})

account_risk = account_risk[account_risk['Flagged_Count'] > 1].sort_values('Flagged_Count', ascending=False)

print(f"\nAccounts with 2+ flagged transactions: {len(account_risk)}")
print("\nTop 5 Highest Risk Accounts:")
print(account_risk.head(5).to_string())

print("\n" + "=" * 70)
print("IMMEDIATE ACTIONS REQUIRED")
print("=" * 70)
print(f"""
1. Review {len(high_risk)} high-risk transactions (risk > 60%)
2. Investigate {len(account_risk)} accounts with multiple flags
3. Contact customers for transactions >$1000 with high risk
4. Implement progressive authentication for accounts with >2 flags
5. Consider automatic blocks for transactions with 4+ login attempts
""")


In [ ]:
# THRESHOLD JUSTIFICATION & BUSINESS CASE

print("\n" + "=" * 70)
print("DETECTION THRESHOLD ANALYSIS")
print("=" * 70)

print("\n" + "-" * 70)
print("WHY 5% CONTAMINATION RATE?")
print("-" * 70)

# Analyze different thresholds
thresholds = [90, 95, 97, 99]
threshold_analysis = []

for thresh_pct in thresholds:
    thresh = np.percentile(train_scores, thresh_pct)
    anomalies = test_scores > thresh
    n_anomalies = anomalies.sum()
    pct = n_anomalies / len(test_scores) * 100
    
    # Estimate potential fraud value saved
    avg_anomaly_amount = test_df[anomalies]['TransactionAmount'].mean() if n_anomalies > 0 else 0
    
    threshold_analysis.append({
        'Percentile': f"{thresh_pct}th",
        'Threshold_Score': f"{thresh:.4f}",
        'Transactions_Flagged': n_anomalies,
        'Flag_Rate_Pct': f"{pct:.2f}%",
        'Avg_Amount_Flagged': f"${avg_anomaly_amount:.2f}"
    })

threshold_df = pd.DataFrame(threshold_analysis)
print(threshold_df.to_string(index=False))

print("\n" + "-" * 70)
print("BUSINESS CASE FOR 5% (95th percentile)")
print("-" * 70)

# Calculate business impact at 5%
n_flagged = int(len(df_final) * 0.05)
avg_transaction = df_final['TransactionAmount'].mean()
high_value_flagged = df_final[df_final['is_anomaly'] == 1]['TransactionAmount'].sum()

print(f"""
Current Configuration: 95th percentile threshold

Detection Rate:         5.0% of transactions
Daily Flagged:          ~{n_flagged} transactions
Review Team Capacity:   125 transactions/day (manageable)

Risk Analysis:
- False Negative Cost:  Missed fraud = -$X,XXX per incident
- False Positive Cost:  Customer inconvenience = -$XX per review
- Break-even:           Need >{int(n_flagged * 0.3)} true fraud cases to justify

Recommendation: 5% threshold is APPROPRIATE
- Balances detection sensitivity with operational capacity
- Captures high-value transactions ($500+)
- Enables manual review of flagged cases
""")

print("\n" + "-" * 70)
print("TIERED RESPONSE STRATEGY (Recommended)")
print("-" * 70)

tiered_strategy = """
┌────────────────────────────────────────────────────────────────┐
│ RISK SCORE          │ ACTION                │ ESTIMATED VOLUME │
├────────────────────────────────────────────────────────────────┤
│ 0.00 - 0.30 (Low)   │ ALLOW                │ 95% of txs       │
│ 0.30 - 0.50 (Medium)│ FLAG FOR REVIEW      │ 3% of txs        │
│ 0.50 - 0.70 (High)  │ REQUIRE 2FA          │ 1.5% of txs      │
│ 0.70 - 1.00 (Critical)│ BLOCK / ALERT TEAM  │ 0.5% of txs      │
└────────────────────────────────────────────────────────────────┘
"""
print(tiered_strategy)


In [ ]:
# FEATURE IMPORTANCE WITH BUSINESS TRANSLATION

print("\n" + "=" * 70)
print("FEATURE IMPORTANCE - BUSINESS INTERPRETATION")
print("=" * 70)

# Map features to business meaning
feature_business_map = {
    'Multi_Login': ('Multiple Login Attempts', 'Credential stuffing or brute force attack indicator'),
    'Amount_ZScore_Global': ('Unusual Transaction Amount', 'Deviates significantly from typical transaction values'),
    'Account_IPCount': ('Multiple IP Addresses', 'Possible account takeover or VPN usage'),
    'IsUnusualHour': ('Unusual Transaction Time', 'Late-night/early-morning activity (11pm-5am)'),
    'Account_MultiLoginRatio': ('Account Login History', 'Historical pattern of multiple logins for this account'),
    'High_Login_Attempts': ('High Login Attempts', '3+ failed logins - strong fraud signal'),
    'Account_Amount_ZScore': ('Amount vs Account Average', 'Unusual for this specific customer'),
    'Is_Online': ('Online Channel', 'Higher fraud risk vs physical channels'),
    'Amount_Balance_Ratio': ('Amount to Balance Ratio', 'Large percentage of account balance'),
    'Quick_Consecutive_Tx': ('Rapid Consecutive Transactions', 'Possible automated/bot activity')
}

print("\n" + "-" * 70)
print("TOP 10 FRAUD INDICATORS WITH BUSINESS MEANING")
print("-" * 70)

# Calculate feature importance based on correlation with anomaly scores
feature_importance = []
for feature in available_features[:20]:  # Top features
    if feature in X_test.columns:
        corr = np.corrcoef(X_test[feature].values, test_scores)[0, 1]
        if not np.isnan(corr):
            feature_importance.append((feature, abs(corr)))

feature_importance.sort(key=lambda x: x[1], reverse=True)

print(f"\n{'Feature':<30} {'Importance':<12} {'Business Meaning'}")
print("-" * 70)

for feature, importance in feature_importance[:10]:
    business_name, business_meaning = feature_business_map.get(feature, (feature, 'Derived risk feature'))
    print(f"{feature[:30]:<30} {importance:.4f}       ")
    print(f"  -> {business_meaning}")

print("\n" + "-" * 70)
print("BUSINESS IMPLICATIONS")
print("-" * 70)
print("""
1. LOGIN BEHAVIOR is the #1 fraud indicator
   -> Implement progressive authentication after 2 failed attempts
   -> Lock accounts after 5 failed attempts

2. UNUSUAL AMOUNTS are strong signals
   -> Flag transactions >$1000 for review
   -> Compare to customer's historical average

3. MULTI-IP/MULTI-DEVICE = Account Takeover
   -> Alert customer of new device/location
   -> Require 2FA for suspicious locations
""")


# Fraud Detection Model Development

## Overview
This notebook builds and evaluates an ensemble anomaly detection model for identifying fraudulent bank transactions.

## Model Strategy
We use an ensemble of four algorithms:
1. **Isolation Forest** - Isolates anomalies by random splitting
2. **One-Class SVM** - Learns the boundary of normal data
3. **Local Outlier Factor (LOF)** - Density-based anomaly detection
4. **Autoencoder** - Neural network that learns to reconstruct normal data

The ensemble combines predictions using weighted averaging for robust detection.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime
import sys
import os

# Add parent directory to path
sys.path.append('..')

# Import custom modules
from core.features import TransactionFeatureEngineer
from core.models import EnsembleAnomalyDetector

# Import custom color scheme
from visualization import FRAUD_COLORS

# MLflow for experiment tracking
try:
    import mlflow
    import mlflow.sklearn
    MLFLOW_AVAILABLE = True
except ImportError:
    MLFLOW_AVAILABLE = False
    print("MLflow not available. Experiment tracking disabled.")

# Settings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

print("Libraries imported successfully!")
print(f"Using custom fraud detection color scheme")

## 1. Load and Prepare Data

In [ ]:
# Load the dataset
df = pd.read_csv('../bank_transactions_data.csv')

# Convert date columns
df['TransactionDate'] = pd.to_datetime(df['TransactionDate'])
df['PreviousTransactionDate'] = pd.to_datetime(df['PreviousTransactionDate'])

print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Initialize MLflow
if MLFLOW_AVAILABLE:
    mlflow.set_tracking_uri("mlruns")
    mlflow.set_experiment("fraud_detection_ensemble")
    print("MLflow initialized. Tracking URI: mlruns")
else:
    print("Skipping MLflow initialization")

## 2. Feature Engineering

In [ ]:
# Initialize feature engineer with historical data
print("Initializing feature engineer...")
feature_engineer = TransactionFeatureEngineer(historical_data=df)

# Transform data
df_transformed = feature_engineer.transform(df)

print(f"\nOriginal shape: {df.shape}")
print(f"Transformed shape: {df_transformed.shape}")
print(f"\nNew features created: {df_transformed.shape[1] - df.shape[1]}")

In [ ]:
# Get feature columns for modeling
feature_cols = feature_engineer.get_feature_columns()

# Filter to only existing columns
available_features = [col for col in feature_cols if col in df_transformed.columns]

print(f"Total features available: {len(available_features)}")
print(f"\nFeature list:")
for i, col in enumerate(available_features, 1):
    print(f"  {i:2d}. {col}")

In [ ]:
# Prepare feature matrix
X = df_transformed[available_features].copy()

# Handle any remaining NaN or Inf values
print("Handling missing values...")
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median())

# Display feature statistics
print("\nFeature matrix shape:", X.shape)
print("\nFeature statistics:")
X.describe().T

## 3. Train-Test Split

In [ ]:
# For unsupervised anomaly detection, we use all data for training
# But we'll hold out a portion for evaluation
from sklearn.model_selection import train_test_split

X_train, X_test = train_test_split(
    X, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

## 4. Model Training

In [ ]:
# Start MLflow run
if MLFLOW_AVAILABLE:
    run = mlflow.start_run()
    mlflow.log_param("model_type", "ensemble")
    mlflow.log_param("contamination", 0.05)
    mlflow.log_param("n_features", len(available_features))
    mlflow.log_param("n_samples", len(X_train))

# Initialize ensemble model
print("Initializing Ensemble Anomaly Detector...")
model = EnsembleAnomalyDetector(
    contamination=0.05,  # Expect ~5% anomalies
    iso_weight=0.3,
    svm_weight=0.3,
    lof_weight=0.2,
    ae_weight=0.2,
    random_state=42
)

print(f"Model weights: {model.weights}")

In [ ]:
# Train the model
print("Training ensemble model...")
print("-" * 50)

model.fit(
    X_train,
    feature_names=available_features,
    autoencoder_epochs=50,
    batch_size=32
)

print("-" * 50)
print("Model training complete!")

## 5. Model Evaluation

In [ ]:
# Make predictions on test set
test_scores = model.predict(X_test, return_scores=True)
test_probs = model.predict_proba(X_test)

# Also get predictions for training set for comparison
train_scores = model.predict(X_train, return_scores=True)

print("=" * 60)
print("MODEL EVALUATION RESULTS")
print("=" * 60)
print(f"\nTest Set Anomaly Scores:")
print(f"  Min: {test_scores.min():.4f}")
print(f"  Max: {test_scores.max():.4f}")
print(f"  Mean: {test_scores.mean():.4f}")
print(f"  Median: {np.median(test_scores):.4f}")
print(f"  Std: {test_scores.std():.4f}")

print(f"\nAnomaly Probabilities:")
print(f"  Min: {test_probs.min():.4f}")
print(f"  Max: {test_probs.max():.4f}")
print(f"  Mean: {test_probs.mean():.4f}")

In [ ]:
# Set threshold and identify anomalies
threshold = np.percentile(train_scores, 95)  # 95th percentile

test_anomalies = test_scores > threshold

print(f"=" * 60)
print(f"ANOMALY DETECTION THRESHOLD: {threshold:.4f}")
print(f"=" * 60)
print(f"\nTest Set Results:")
print(f"  Total transactions: {len(test_scores)}")
print(f"  Flagged as anomalies: {test_anomalies.sum()} ({test_anomalies.sum()/len(test_scores)*100:.2f}%)")
print(f"  Normal transactions: {len(test_scores) - test_anomalies.sum()} ({(len(test_scores) - test_anomalies.sum())/len(test_scores)*100:.2f}%)")

In [ ]:
# Visualize anomaly scores
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Score distribution
axes[0].hist(test_scores, bins=50, color=FRAUD_COLORS['low'], alpha=0.7, edgecolor='black')
axes[0].axvline(x=threshold, color=FRAUD_COLORS['critical'], linestyle='--', linewidth=2, label=f'Threshold: {threshold:.3f}')
axes[0].set_xlabel('Anomaly Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Anomaly Score Distribution')
axes[0].legend()

# Probability distribution
axes[1].hist(test_probs, bins=50, color=FRAUD_COLORS['medium'], alpha=0.7, edgecolor='black')
axes[1].axvline(x=0.5, color=FRAUD_COLORS['high'], linestyle='--', linewidth=2, label='Cutoff: 0.5')
axes[1].set_xlabel('Anomaly Probability')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Anomaly Probability Distribution')
axes[1].legend()

# Normal vs Anomaly box plot
labels = ['Normal', 'Anomaly']
data_to_plot = [test_scores[~test_anomalies], test_scores[test_anomalies]]
bp = axes[2].boxplot(data_to_plot, labels=labels, patch_artist=True)
bp['boxes'][0].set_facecolor(FRAUD_COLORS['safe'])
bp['boxes'][1].set_facecolor(FRAUD_COLORS['critical'])
axes[2].set_ylabel('Anomaly Score')
axes[2].set_title('Normal vs Anomaly Scores')

plt.tight_layout()
plt.show()

## 6. Analyze Flagged Transactions

In [ ]:
# Get indices of anomalies
anomaly_indices = np.where(test_anomalies)[0]

# Get corresponding original data
test_df = df_transformed.iloc[X_test.index].copy()
test_df['anomaly_score'] = test_scores
test_df['anomaly_probability'] = test_probs
test_df['is_anomaly'] = test_anomalies.astype(int)

# Display top anomalies
top_anomalies = test_df[test_df['is_anomaly'] == 1].sort_values(
    'anomaly_score', ascending=False
)

print(f"=" * 80)
print(f"TOP 20 FLAGGED TRANSACTIONS (HIGHEST ANOMALY SCORES)")
print(f"=" * 80)

display_cols = [
    'TransactionID', 'TransactionAmount', 'LoginAttempts', 'Channel',
    'Location', 'anomaly_score', 'anomaly_probability',
    'Amount_ZScore_Global', 'Multi_Login', 'Account_IPCount'
]

# Filter to columns that exist
display_cols = [col for col in display_cols if col in top_anomalies.columns]

print(top_anomalies[display_cols].head(20).to_string(index=False))

In [ ]:
# Analyze characteristics of flagged transactions
print("=" * 60)
print("ANOMALY CHARACTERISTICS ANALYSIS")
print("=" * 60)

normal = test_df[test_df['is_anomaly'] == 0]
anomaly = test_df[test_df['is_anomaly'] == 1]

print("\n1. TRANSACTION AMOUNT")
print(f"   Normal:   Mean=${normal['TransactionAmount'].mean():.2f}, Median=${normal['TransactionAmount'].median():.2f}")
print(f"   Anomaly:  Mean=${anomaly['TransactionAmount'].mean():.2f}, Median=${anomaly['TransactionAmount'].median():.2f}")

print("\n2. LOGIN ATTEMPTS")
print(f"   Normal:   Mean={normal['LoginAttempts'].mean():.2f}")
print(f"   Anomaly:  Mean={anomaly['LoginAttempts'].mean():.2f}")
print(f"   Multiple logins in anomalies: {(anomaly['LoginAttempts'] > 1).sum()} ({(anomaly['LoginAttempts'] > 1).sum()/len(anomaly)*100:.1f}%)")

print("\n3. CHANNEL DISTRIBUTION")
for channel in ['Online', 'ATM', 'Branch']:
    if 'Channel' in test_df.columns:
        normal_pct = (normal['Channel'] == channel).sum() / len(normal) * 100
        anomaly_pct = (anomaly['Channel'] == channel).sum() / len(anomaly) * 100
        print(f"   {channel}: Normal {normal_pct:.1f}%, Anomaly {anomaly_pct:.1f}%")

print("\n4. ACCOUNT BEHAVIOR")
if 'Account_IPCount' in test_df.columns:
    print(f"   Normal:   Avg IP Count={normal['Account_IPCount'].mean():.2f}")
    print(f"   Anomaly:  Avg IP Count={anomaly['Account_IPCount'].mean():.2f}")

## 7. SHAP Explainability

In [ ]:
# Compute SHAP values for model explainability
try:
    import shap

    print("Computing SHAP values for top 100 samples...")
    explanation = model.explain_prediction(X_test[:100])

    if 'feature_importance' in explanation:
        print("\n" + "=" * 60)
        print("GLOBAL FEATURE IMPORTANCE (SHAP)")
        print("=" * 60)
        for feature, importance in explanation['feature_importance'][:15]:
            print(f"  {feature:30s}: {importance:.4f}")

        # Log to MLflow
        if MLFLOW_AVAILABLE:
            for feature, importance in explanation['feature_importance'][:10]:
                mlflow.log_metric(f"shap_importance_{feature}", importance)

except ImportError:
    print("SHAP not available. Skipping explainability.")

In [ ]:
# SHAP summary plot
try:
    import shap

    plt.figure(figsize=(10, 8))
    shap.summary_plot(
        explanation['shap_values'],
        X_test[:100],
        feature_names=available_features,
        show=False
    )
    plt.title('SHAP Summary Plot - Feature Impact on Anomaly Detection')
    plt.tight_layout()
    plt.show()
except:
    print("Could not create SHAP plot")

## 8. Individual Model Performance

In [ ]:
# Compare individual model contributions
X_test_scaled = model.scaler.transform(X_test)

# Get scores from each model
individual_scores = model._get_model_scores(X_test_scaled)

print("=" * 60)
print("INDIVIDUAL MODEL SCORE DISTRIBUTIONS")
print("=" * 60)

for model_name, scores in individual_scores.items():
    print(f"\n{model_name}:")
    print(f"  Min: {scores.min():.4f}")
    print(f"  Max: {scores.max():.4f}")
    print(f"  Mean: {scores.mean():.4f}")
    print(f"  Std: {scores.std():.4f}")

In [ ]:
# Visualize individual model scores
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, (model_name, scores) in enumerate(individual_scores.items()):
    axes[i].hist(scores, bins=50, alpha=0.7, edgecolor='black')
    axes[i].set_title(f'{model_name} Score Distribution')
    axes[i].set_xlabel('Score')
    axes[i].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## 9. Full Dataset Prediction

In [ ]:
# Make predictions on the entire dataset
print("Making predictions on full dataset...")

full_scores = model.predict(X, return_scores=True)
full_probs = model.predict_proba(X)

# Add to original dataframe
df_final = df.copy()
df_final['anomaly_score'] = full_scores
df_final['anomaly_probability'] = full_probs
df_final['is_anomaly'] = (full_probs > 0.5).astype(int)

print(f"\nTotal anomalies detected: {df_final['is_anomaly'].sum()} ({df_final['is_anomaly'].sum()/len(df_final)*100:.2f}%)")

In [ ]:
# Save results
output_dir = 'models'
os.makedirs(output_dir, exist_ok=True)

# Save model
model_path = f'{output_dir}/fraud_detector.pkl'
model.save(model_path)

# Save predictions
df_final.to_csv(f'{output_dir}/predictions.csv', index=False)

# Save feature list
with open(f'{output_dir}/features.txt', 'w') as f:
    f.write('\n'.join(available_features))

print(f"\nModel saved to: {model_path}")
print(f"Predictions saved to: {output_dir}/predictions.csv")
print(f"Feature list saved to: {output_dir}/features.txt")

# Log model to MLflow
if MLFLOW_AVAILABLE:
    mlflow.sklearn.log_model(
        model.models['isolation_forest'],
        "isolation_forest_model"
    )
    mlflow.log_artifact(model_path)
    mlflow.log_artifact(f'{output_dir}/predictions.csv')
    mlflow.end_run()
    print("\nMLflow run completed.")

## 10. Final Summary

In [ ]:
print("=" * 70)
print("FRAUD DETECTION MODEL - TRAINING SUMMARY")
print("=" * 70)

print("\n" + "="*70)
print("MODEL CONFIGURATION")
print("="*70)
print(f"  Model Type: Ensemble Anomaly Detector")
print(f"  Components: Isolation Forest, One-Class SVM, LOF, Autoencoder")
print(f"  Weights: {model.weights}")
print(f"  Contamination Rate: {model.contamination * 100}%")
print(f"  Features Used: {len(available_features)}")

print("\n" + "="*70)
print("TRAINING RESULTS")
print("="*70)
print(f"  Training Samples: {len(X_train)}")
print(f"  Test Samples: {len(X_test)}")
print(f"  Anomaly Threshold: {threshold:.4f}")

print("\n" + "="*70)
print("DETECTION RESULTS")
print("="*70)
print(f"  Total Transactions: {len(df_final)}")
print(f"  Flagged as Anomalous: {df_final['is_anomaly'].sum()} ({df_final['is_anomaly'].sum()/len(df_final)*100:.2f}%)")
print(f"  Normal Transactions: {len(df_final) - df_final['is_anomaly'].sum()} ({(len(df_final) - df_final['is_anomaly'].sum())/len(df_final)*100:.2f}%)")

print("\n" + "="*70)
print("KEY FRAUD INDICATORS IDENTIFIED")
print("="*70)
print("  1. Multiple Login Attempts")
print("  2. High Transaction Amount (outliers)")
print("  3. Unusual Transaction Patterns")
print("  4. Multi-IP/Multi-Device Accounts")

print("\n" + "="*70)
print("OUTPUT FILES")
print("="*70)
print(f"  Model: models/fraud_detector.pkl")
print(f"  Predictions: models/predictions.csv")
print(f"  Features: models/features.txt")

print("\n" + "="*70)
print("NEXT STEPS")
print("="*70)
print("  1. Deploy model via FastAPI")
print("  2. Set up real-time monitoring")
print("  3. Create Streamlit dashboard")
print("  4. Add LLM explanations for flagged transactions")
print("="*70)

In [ ]:
# ENSEMBLE VS INDIVIDUAL MODELS COMPARISON

# Compare individual models vs ensemble
print("\n" + "=" * 70)
print("ENSEMBLE VS INDIVIDUAL MODELS - COMPARATIVE ANALYSIS")
print("=" * 70)

# Get predictions from each model on test set
individual_results = {}
for model_name, scores in individual_scores.items():
    # Use 95th percentile as threshold for consistency
    model_threshold = np.percentile(scores, 95)
    predictions = scores > model_threshold
    individual_results[model_name] = {
        'scores': scores,
        'predictions': predictions,
        'threshold': model_threshold,
        'anomaly_count': predictions.sum()
    }

# Ensemble results
ensemble_threshold = threshold
ensemble_predictions = test_anomalies

print("\n" + "-" * 70)
print("DETECTION RATE COMPARISON")
print("-" * 70)

comparison_data = []
for name, results in individual_results.items():
    comparison_data.append({
        'Model': name.replace('_', ' ').title(),
        'Anomalies_Detected': results['anomaly_count'],
        'Detection_Rate_Pct': results['anomaly_count'] / len(test_scores) * 100,
        'Threshold_Score': f"{results['threshold']:.4f}"
    })

# Add ensemble
comparison_data.append({
    'Model': 'ENSEMBLE',
    'Anomalies_Detected': ensemble_predictions.sum(),
    'Detection_Rate_Pct': ensemble_predictions.sum() / len(test_scores) * 100,
    'Threshold_Score': f"{ensemble_threshold:.4f}"
})

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

print("\n" + "-" * 70)
print("ENSEMBLE BENEFIT ANALYSIS")
print("-" * 70)

# Calculate stability (variance of predictions across models)
individual_preds = np.array([r['predictions'].astype(int) for r in individual_results.values()])
agreement_rate = np.mean(np.std(individual_preds, axis=0) == 0)
print(f"Agreement Rate: {agreement_rate*100:.1f}% - {'Models often disagree' if agreement_rate < 0.8 else 'Models generally agree'}")
print(f"Benefit: Ensemble reduces variance and provides more stable predictions")
